# Does AI Have Taste? — run everything here

**Before starting:** `Runtime → Change runtime type → T4 GPU`.

Run cells top to bottom. Every step saves its output to Google Drive (`MyDrive/fashion-taste-work`)
and **skips or resumes** work that is already done, so if Colab disconnects, just run the setup cells
(1–4) again and continue where you left off.

| Step | Cell | Time on a T4 | Who |
|---|---|---|---|
| Setup + data | 1–5 | ~15 min (first time) | anyone |
| CLIP embeddings + attributes | 6–7 | ~20 min | anyone |
| Trick outfits + validation booklet | 8 | ~1 min | anyone |
| Train head / full fine-tune | 9–10 | ~10 min / several hours | anyone |
| Survey | 11 | ~1 min | survey owner |
| Gemini (4 prompting strategies) | 12 | days on free tier (resumable) | Gemini owner |
| Results, figures, demo | 13–17 | minutes | anyone |

## 1. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L || echo "No GPU! Runtime -> Change runtime type -> T4 GPU"

## 2. Get the code
If the repo is **private**, add a GitHub token (Settings → Developer settings → Fine-grained token, read access to this repo)
in Colab's 🔑 **Secrets** panel as `GITHUB_TOKEN`. If it's public, nothing is needed.

In [ ]:
import os, subprocess
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f"https://{tok}@github.com/riyaghate/fashion-taste.git" if tok else "https://github.com/riyaghate/fashion-taste.git"
if os.path.exists('/content/fashion-taste'):
    !cd /content/fashion-taste && git pull -q
else:
    subprocess.run(['git', 'clone', '-q', url, '/content/fashion-taste'], check=True)
%cd /content/fashion-taste
!git log -1 --oneline

## 3. Install packages

In [ ]:
!pip install -q -r requirements.txt

## 4. Paths (run this cell after every reconnect)

In [ ]:
import os
os.environ['FT_DATA'] = '/content/data/polyvore_outfits'          # fast local disk
os.environ['FT_WORK'] = '/content/drive/MyDrive/fashion-taste-work'  # everything we produce
os.environ['FT_SPLIT'] = 'nondisjoint'
os.makedirs(os.environ['FT_WORK'], exist_ok=True)

## 5. Get the Polyvore Outfits data (~6 GB)
**Once, by hand:** open https://github.com/mvasil/fashion-compatibility, click the dataset download link ("from here (6G)").
It opens Google Drive: right-click the file → **Organize → Add shortcut** to `My Drive` (or download and upload it).
Set `TAR` below to where it is. If Google says "too many downloads", the shortcut route still works.
The cell copies it to Colab's local disk and unpacks it (~10 min; needed again after each full reset).

In [ ]:
import os, glob
TAR = '/content/drive/MyDrive/polyvore_outfits.tar.gz'   # <- change if yours is elsewhere
if not os.path.exists(os.environ['FT_DATA'] + '/polyvore_item_metadata.json'):
    if not os.path.exists(TAR):
        print('Not found:', TAR); print('Candidates:', glob.glob('/content/drive/MyDrive/**/*polyvore*', recursive=True)[:10])
    else:
        !mkdir -p /content/data && cp "$TAR" /content/data/p.tar.gz && tar -xzf /content/data/p.tar.gz -C /content/data && rm /content/data/p.tar.gz
        # some versions unpack to a differently named folder:
        found = glob.glob('/content/data/**/polyvore_item_metadata.json', recursive=True)
        if found and os.path.dirname(found[0]) != os.environ['FT_DATA']:
            os.environ['FT_DATA'] = os.path.dirname(found[0]); print('using', os.environ['FT_DATA'])
!python -m fashion_taste.data

## 6. Zero-shot CLIP embeddings for every item

In [ ]:
!python -m fashion_taste.embed

## 7. Tag items with color / pattern / formality / season
Keyword rules and CLIP must **agree** for an attribute to count. The agreement table is a result for the report.

In [ ]:
!python -m fashion_taste.attributes

## 8. Build single-rule trick outfits (hard negatives) + the blind validation booklet
If any rule gets few examples, re-run with `--attr_mode keyword` and say so in the report.

**Validation (2 teammates, ~45 min each):** download `validation/validation_booklet.html` and
`validation_labels_TEMPLATE.csv` from Drive. Each labeler copies the template to
`validation_labels_<name>.csv`, fills a label for every case **without discussing**, and uploads it back to the same folder.
Don't open the `..._DO_NOT_OPEN_UNTIL_DONE.csv` key.

In [ ]:
!python -m fashion_taste.hard_negatives build
!python -m fashion_taste.hard_negatives booklet

## 9. Train a small head on frozen CLIP features (~10 min)

In [ ]:
!python -m fashion_taste.train head

## 10. Fully fine-tune CLIP (long — resumable)
Runs up to `--max_hours` per session, then stops cleanly; run the cell again to continue.
Keep this tab open (Colab disconnects idle tabs). If you run out of GPU time, `--unfreeze_last 4` trains only
the last 4 vision blocks and is much faster (report it as a partial fine-tune).

In [ ]:
!python -m fashion_taste.train finetune --epochs 5 --max_hours 3

## 11. Build the human survey
1. Run the cell. It saves outfit images to Drive in `fashion-taste-work/survey/fashion_taste_survey_images/`.
2. Open https://script.google.com → **New project**, paste the contents of `survey/make_form.gs` (printed below), click **Run**, allow access.
3. The execution log prints the link to share. **Send it out early** (target: 20+ responses).
4. When responses are in: Form → Responses → **Link to Sheets** → File → Download → CSV, save it as `fashion-taste-work/survey/responses.csv`.

In [ ]:
!python -m fashion_taste.survey build
print(open(os.environ['FT_WORK'] + '/survey/make_form.gs').read())

## 12. Gemini, 4 prompting strategies (resumable across days)
Get a free key at https://aistudio.google.com/apikey and add it in 🔑 **Secrets** as `GEMINI_API_KEY`.
Check the model list in AI Studio and set `GEMINI_MODEL` to a vision-capable model available on your tier.

Run the **dry run** first and look at the example prompt images. Then run `direct` first (needed for every comparison),
then the others. When the daily quota runs out, the cell stops and says so; **run it again the next day**.
If you see many `429` errors, lower `--rpm`.

In [ ]:
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_MODEL'] = 'gemini-2.5-flash'   # <- change to a model your key can use
!python -m fashion_taste.vlm --dry_run --strategies direct,fewshot

In [ ]:
!python -m fashion_taste.vlm --strategies direct,reason,rubric,fewshot --rpm 8

## 13. Check the trick outfits against the two labelers (after both label files are uploaded)

In [ ]:
!python -m fashion_taste.hard_negatives validate

## 14. Survey analysis (after responses.csv is in place)

In [ ]:
!python -m fashion_taste.survey analyze

## 15. All results tables (re-run whenever more Gemini answers come in)

In [ ]:
!python -m fashion_taste.evaluate

## 16. Figures (PNG for slides, PDF for the LaTeX report)

In [ ]:
!python -m fashion_taste.plots
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(os.environ['FT_WORK'] + '/figures/*.png')):
    print(f); display(Image(f, width=700))

## 17. "Rate My Fit" demo
Photograph each item **separately**, flat on a plain surface, good light (iPhone: Settings → Camera → Formats →
*Most Compatible* so photos are JPEG). Upload to Drive as:
```
fashion-taste-work/demo_photos/<outfit_name>/<item>.jpg
fashion-taste-work/demo_closet/{tops,bottoms,shoes}/<item>.jpg   (optional: "pick my best outfit")
```

In [ ]:
W = os.environ['FT_WORK']
!python -m fashion_taste.demo rate --photos "$W/demo_photos" --gemini
import pandas as pd; display(pd.read_csv(W + '/demo_out/ratings.csv'))

In [ ]:
!python -m fashion_taste.demo closet --photos "$W/demo_closet"
for f in sorted(glob.glob(W + '/demo_out/closet_*.jpg')): display(Image(f, width=600))